# 06 · Clustering con K-Means

> **Objetivo:** entrenar K-Means, elegir el k óptimo, y persistir el modelo.

## ¿Qué vamos a hacer?

1. Recapitular cómo funciona K-Means.
2. Aplicar el método del codo y la silueta para elegir `k`.
3. Entrenar el modelo final.
4. Visualizar los clusters en 2D (PCA).
5. Guardar el modelo.

## Concepto teórico

K-Means itera dos pasos:

1. **Asignación**: cada punto se asigna al **centroide** más cercano.
2. **Actualización**: cada centroide se mueve al centro de sus puntos.

Repite hasta que los centroides dejan de moverse. Minimiza la **inertia**:

$$\text{Inertia} = \sum_{i=1}^{n} \min_{c \in C} \|x_i - c\|^2$$

### Fortalezas
- Rápido (escalable a datasets grandes).
- Determinístico con `random_state`.
- Fácil de interpretar (centroides = "cliente promedio" del cluster).

### Limitaciones
- Necesitas elegir `k` manualmente.
- Asume clusters **esféricos** y de tamaño **similar**.
- Sensible a outliers (los centroides se ven jalados).
- Sensible a la escala → siempre escalar antes.


In [ ]:
# Permite importar el paquete src/ desde el notebook
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from src.config import (
    FEATURES_DATA_FILE,
    EXTENDED_NUMERIC_FEATURES,
    KMEANS_K_RANGE,
    KMEANS_MODEL_FILE,
    PIPELINE_FILE,
    SEGMENTS_FILE,
    RANDOM_STATE,
)
from src.features.preprocessing import build_preprocessing_pipeline
from src.models.clustering import find_optimal_k, fit_kmeans
from src.visualization.plots import plot_clusters_2d, plot_elbow_and_silhouette
from sklearn.decomposition import PCA


## 1. Cargar features y aplicar pipeline

In [ ]:
features = pd.read_parquet(FEATURES_DATA_FILE)
print(f"Clientes: {len(features):,}")

pipeline = build_preprocessing_pipeline(
    numeric_features=EXTENDED_NUMERIC_FEATURES,
    use_log=True,
)
X = pipeline.fit_transform(features[EXTENDED_NUMERIC_FEATURES])
print(f"Matriz de features: {X.shape}")


## 2. Búsqueda del K óptimo

Probamos `k` de 2 a 10 y graficamos inertia (codo) y silueta.


In [ ]:
metrics = find_optimal_k(X, k_range=KMEANS_K_RANGE, random_state=RANDOM_STATE)
metrics


In [ ]:
fig = plot_elbow_and_silhouette(metrics)
plt.show()


**Cómo leer estas curvas:**

- **Codo (inertia):** busca el "codo" donde la pendiente se aplana.
  Pasar de k=3 a k=4 te da una mejora grande; pasar de k=8 a k=9 te da
  una mejora marginal.
- **Silueta:** picos altos indican clusters bien definidos. Si k=4 tiene
  silueta=0.45 y k=5 tiene 0.48, la diferencia puede no ser significativa.

> **Tip:** no elijas el k que maximiza la silueta a ciegas. **K=2** suele
> ganar siempre porque dividir el dataset en dos mitades grandes da
> alta silueta, pero rara vez tiene sentido de negocio.

## 3. Davies-Bouldin y Calinski-Harabasz


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(metrics.index, metrics["davies_bouldin"], "o-", color="purple")
axes[0].set_title("Davies-Bouldin (más BAJO mejor)")
axes[0].set_xlabel("k"); axes[0].set_ylabel("DB Index")
axes[0].grid(True, alpha=0.3)

axes[1].plot(metrics.index, metrics["calinski_harabasz"], "o-", color="teal")
axes[1].set_title("Calinski-Harabasz (más ALTO mejor)")
axes[1].set_xlabel("k"); axes[1].set_ylabel("CH Index")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Decisión de k

Combinando las cuatro métricas + sentido de negocio:

- Para Online Retail II, **k=4** suele ser un compromiso razonable.
- Da segmentos accionables: Champions, Leales, En Riesgo, Perdidos.
- Más k aumenta la complejidad de la estrategia de marketing sin
  ganar mucho en homogeneidad.

> **Tip:** la decisión final debe consensuarse con el equipo de marketing.
> El "mejor k" depende de cuántos segmentos el negocio puede activar.

## 5. Entrenar el modelo final con k = 4


In [ ]:
K_FINAL = 4
kmeans = fit_kmeans(X, n_clusters=K_FINAL, random_state=RANDOM_STATE)

features["cluster_kmeans"] = kmeans.labels_
print("Distribución de clientes por cluster:")
print(features["cluster_kmeans"].value_counts().sort_index())


## 6. Visualización 2D con PCA


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X)
print(f"Varianza explicada: {pca.explained_variance_ratio_.sum():.2%}")

fig = plot_clusters_2d(X_2d, kmeans.labels_, title=f"K-Means · k={K_FINAL}")
plt.show()


## 7. Centroides en escala original

Los centroides en el espacio escalado no son interpretables. Para entender
"cómo es" cada cluster, calculamos el promedio de las features originales
por cluster.


In [ ]:
profile_kmeans = features.groupby("cluster_kmeans")[EXTENDED_NUMERIC_FEATURES].mean().round(2)
profile_kmeans["n_customers"] = features["cluster_kmeans"].value_counts().sort_index()
profile_kmeans


> Interpreta cada cluster: ¿quién tiene la Recency más baja?
> ¿Quién tiene la Monetary más alta? Esos son tus Champions.

## 8. Guardar el modelo y los segmentos


In [ ]:
joblib.dump(kmeans, KMEANS_MODEL_FILE)
joblib.dump(pipeline, PIPELINE_FILE)
features.reset_index().to_parquet(SEGMENTS_FILE, index=False)
print(f"Modelo:    {KMEANS_MODEL_FILE}")
print(f"Pipeline:  {PIPELINE_FILE}")
print(f"Segmentos: {SEGMENTS_FILE}")


## Resumen

- K-Means minimiza la inertia → busca clusters esféricos.
- El **codo** y la **silueta** ayudan a elegir `k`, pero no sustituyen
  el juicio de negocio.
- Los centroides son interpretables si los devuelves a la escala original.

---

## Preguntas de Reflexión

1. ¿Por qué la silueta para `k=2` es casi siempre la más alta? ¿Por qué
   raramente es la decisión correcta?
2. Si `random_state` no estuviera fijado, ¿qué tan estables serían los clusters?
3. ¿Qué cluster esperarías que sea más sensible a una promoción?
4. ¿Qué pasaría si tuvieras un cliente con `Frequency=200` y `Monetary=500000`?
   ¿En qué cluster terminaría? ¿Distorsionaría el centroide?

> **Próximo paso:** ``07_clustering_dbscan.ipynb`` — un enfoque distinto
> que detecta outliers automáticamente.
